# 05 - Transformer Architecture Walkthrough

**Paper:** "Attention Is All You Need" (Vaswani et al., 2017)

This notebook walks through the **complete Transformer architecture**, building each component from scratch and assembling them into a working model.

We'll cover:
1. Architecture overview (Figure 1)
2. Positional Encoding
3. Encoder block
4. Decoder block
5. Full Transformer assembly
6. Key insights from the paper

**Prerequisites:** Notebooks 01-04

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

torch.manual_seed(42)
print("Ready!")

## 1. Architecture Overview (Paper Figure 1)

```
                    ┌─────────────────────────┐
                    │    Output Probabilities  │
                    │      (Linear + Softmax)  │
                    └────────────┬────────────┘
                                 │
                    ┌────────────┴────────────┐
                    │                         │
                    │   DECODER (×N=6)        │
                    │                         │
                    │   ┌─────────────────┐   │
                    │   │  Feed Forward    │   │
                    │   │  Add & Norm      │   │
                    │   ├─────────────────┤   │
            ┌───────┤   │  Cross-Attention │   │
            │       │   │  (Q from decoder,│   │
            │       │   │   K,V from enc)  │   │
            │       │   │  Add & Norm      │   │
            │       │   ├─────────────────┤   │
            │       │   │  Masked Self-Attn│   │
            │       │   │  Add & Norm      │   │
            │       │   └─────────────────┘   │
            │       └────────────┬────────────┘
            │                    │
  ┌─────────┴─────────┐  ┌──────┴──────┐
  │                   │  │             │
  │  ENCODER (×N=6)   │  │  Output     │
  │                   │  │  Embedding  │
  │  ┌─────────────┐  │  │     +       │
  │  │ Feed Forward │  │  │  Positional│
  │  │ Add & Norm   │  │  │  Encoding  │
  │  ├─────────────┤  │  └──────┬──────┘
  │  │ Self-Attn    │  │         │
  │  │ Add & Norm   │  │   Output Tokens
  │  └─────────────┘  │   (shifted right)
  └─────────┬─────────┘
            │
    ┌───────┴───────┐
    │    Input      │
    │  Embedding +  │
    │  Positional   │
    │  Encoding     │
    └───────┬───────┘
            │
      Input Tokens
```

Key components to build:
- **Positional Encoding** — inject position information
- **Multi-Head Attention** — the core mechanism (from notebook 04)
- **Feed-Forward Network** — position-wise MLP
- **Add & Norm** — residual connections + layer normalization

## 2. Positional Encoding

**Problem:** Attention treats input as a **set** — it has no notion of order! "cat sat on mat" and "mat sat on cat" would look the same.

**Solution:** Add position information to the embeddings.

The paper uses sinusoidal positional encodings (Section 3.5):

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

Why sinusoidal? The paper hypothesizes it allows the model to learn to attend to **relative positions**, because $PE_{pos+k}$ can be represented as a linear function of $PE_{pos}$.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()  # (max_len, 1)
        
        # Compute the division term: 10000^(2i/d_model)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)  # even dimensions
        pe[:, 1::2] = torch.cos(position * div_term)  # odd dimensions
        
        pe = pe.unsqueeze(0)  # (1, max_len, d_model) for broadcasting
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        # x shape: (batch, seq_len, d_model)
        return x + self.pe[:, :x.size(1)]

# Test it
d_model = 64
pe = PositionalEncoding(d_model)
x = torch.zeros(1, 20, d_model)  # 20 positions
encoded = pe(x)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {encoded.shape}")
print(f"PE values at position 0: {encoded[0, 0, :8].tolist()}")

In [ ]:
# Visualize positional encodings
pe_vis = PositionalEncoding(d_model=64, max_len=100)
pe_values = pe_vis.pe[0, :100, :].numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap of all positions × dimensions
im = axes[0].imshow(pe_values, cmap='RdBu', aspect='auto')
axes[0].set_xlabel('Dimension')
axes[0].set_ylabel('Position')
axes[0].set_title('Positional Encoding Heatmap')
plt.colorbar(im, ax=axes[0])

# Show that nearby positions are similar
positions = [0, 1, 2, 3, 50, 99]
sim_matrix = np.zeros((len(positions), len(positions)))
for i, p1 in enumerate(positions):
    for j, p2 in enumerate(positions):
        v1, v2 = pe_values[p1], pe_values[p2]
        sim_matrix[i, j] = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

im2 = axes[1].imshow(sim_matrix, cmap='YlOrRd', vmin=-1, vmax=1)
axes[1].set_xticks(range(len(positions)))
axes[1].set_yticks(range(len(positions)))
axes[1].set_xticklabels(positions)
axes[1].set_yticklabels(positions)
axes[1].set_title('Cosine Similarity Between Positions')
for i in range(len(positions)):
    for j in range(len(positions)):
        axes[1].text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center', fontsize=9)
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()
print("Nearby positions (0,1,2,3) have high similarity.")
print("Distant positions (0 vs 99) have low similarity.")

## 3. Multi-Head Attention (from Notebook 04)

We'll reuse this component. Quick recap: project Q, K, V into h heads, run scaled dot-product attention in parallel, concatenate, project back.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
    
    def forward(self, Q, K, V, mask=None):
        batch_size = Q.shape[0]
        
        Q = self.W_q(Q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(K).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(V).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        scores = Q @ K.transpose(-2, -1) / (self.d_k ** 0.5)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1)
        context = weights @ V
        
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.W_o(context)

print("MultiHeadAttention ready ✓")

## 4. Feed-Forward Network

**Paper Section 3.3:** A simple two-layer MLP applied to each position independently:

$$\text{FFN}(x) = \max(0, xW_1 + b_1)W_2 + b_2$$

- Inner dimension: $d_{ff} = 2048$ (4× the model dimension)
- This is where a lot of the model's "knowledge" is stored

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        return self.linear2(self.relu(self.linear1(x)))

# Test
d_model = 64
d_ff = 256
ff = FeedForward(d_model, d_ff)
x = torch.randn(1, 10, d_model)
out = ff(x)
print(f"Input:  {x.shape}")
print(f"Output: {out.shape}  (same shape — position-wise)")
print(f"FFN params: {sum(p.numel() for p in ff.parameters()):,}")

## 5. Encoder Block

Each encoder block has:
1. **Multi-Head Self-Attention** + Add & LayerNorm
2. **Feed-Forward Network** + Add & LayerNorm

**Residual connections** ("Add"): $\text{output} = \text{LayerNorm}(x + \text{SubLayer}(x))$

Why residual connections? They let gradients flow directly through the network, making deep networks trainable.

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Sub-layer 1: Self-attention + Add & Norm
        attn_out = self.self_attn(x, x, x, mask)  # Q=K=V=x
        x = self.norm1(x + self.dropout(attn_out))
        
        # Sub-layer 2: FFN + Add & Norm
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))
        
        return x

# Test
enc_block = EncoderBlock(d_model=64, num_heads=8, d_ff=256)
x = torch.randn(2, 10, 64)  # batch=2, seq=10, d_model=64
out = enc_block(x)
print(f"Input:  {x.shape}")
print(f"Output: {out.shape}  (same shape!)")
print(f"\nEncoder block params: {sum(p.numel() for p in enc_block.parameters()):,}")

In [ ]:
# Stack N=6 encoder blocks (as in the paper)
class Encoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            EncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
    
    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer(x, mask)
        return x

encoder = Encoder(d_model=64, num_heads=8, d_ff=256, num_layers=6)
x = torch.randn(2, 10, 64)
out = encoder(x)
print(f"6-layer encoder: {x.shape} → {out.shape}")
print(f"Total encoder params: {sum(p.numel() for p in encoder.parameters()):,}")

## 6. Decoder Block

The decoder block has **three** sub-layers (one more than encoder):
1. **Masked Multi-Head Self-Attention** — can't look at future tokens
2. **Cross-Attention** — queries from decoder, keys/values from encoder
3. **Feed-Forward Network**

Each with residual connection + layer norm.

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.masked_self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        # Sub-layer 1: Masked self-attention
        attn_out = self.masked_self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_out))
        
        # Sub-layer 2: Cross-attention (Q from decoder, K/V from encoder)
        cross_out = self.cross_attn(x, encoder_output, encoder_output, src_mask)
        x = self.norm2(x + self.dropout(cross_out))
        
        # Sub-layer 3: FFN
        ffn_out = self.ffn(x)
        x = self.norm3(x + self.dropout(ffn_out))
        
        return x

class Decoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            DecoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
    
    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)
        return x

# Test
d_model = 64
decoder = Decoder(d_model=d_model, num_heads=8, d_ff=256, num_layers=6)
enc_out = torch.randn(2, 10, d_model)  # encoder output
dec_input = torch.randn(2, 8, d_model)  # decoder input (target sequence)

# Create causal mask for decoder
tgt_len = 8
tgt_mask = torch.tril(torch.ones(tgt_len, tgt_len)).unsqueeze(0).unsqueeze(0)  # (1, 1, tgt_len, tgt_len)

dec_out = decoder(dec_input, enc_out, tgt_mask=tgt_mask)
print(f"Encoder output: {enc_out.shape}")
print(f"Decoder input:  {dec_input.shape}")
print(f"Decoder output: {dec_out.shape}")
print(f"Total decoder params: {sum(p.numel() for p in decoder.parameters()):,}")

## 7. Full Transformer

Now we assemble everything:

**Encoder side:** Input tokens → Embedding → + Positional Encoding → Encoder Stack  
**Decoder side:** Output tokens → Embedding → + Positional Encoding → Decoder Stack → Linear → Softmax

In [ ]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512,
                 num_heads=8, d_ff=2048, num_layers=6, dropout=0.1, max_len=5000):
        super().__init__()
        
        # Embeddings
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_len)
        
        # Encoder and Decoder
        self.encoder = Encoder(d_model, num_heads, d_ff, num_layers, dropout)
        self.decoder = Decoder(d_model, num_heads, d_ff, num_layers, dropout)
        
        # Final linear layer to vocab
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)
        
        self.d_model = d_model
        self.dropout = nn.Dropout(dropout)
    
    def encode(self, src, src_mask=None):
        x = self.src_embedding(src) * math.sqrt(self.d_model)  # scale embeddings
        x = self.positional_encoding(x)
        x = self.dropout(x)
        return self.encoder(x, src_mask)
    
    def decode(self, tgt, encoder_output, src_mask=None, tgt_mask=None):
        x = self.tgt_embedding(tgt) * math.sqrt(self.d_model)
        x = self.positional_encoding(x)
        x = self.dropout(x)
        return self.decoder(x, encoder_output, src_mask, tgt_mask)
    
    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        encoder_output = self.encode(src, src_mask)
        decoder_output = self.decode(tgt, encoder_output, src_mask, tgt_mask)
        logits = self.fc_out(decoder_output)
        return logits

print("Transformer class defined ✓")

In [ ]:
# Create a Transformer and run a forward pass
src_vocab = 1000   # source vocabulary size
tgt_vocab = 1200   # target vocabulary size
d_model = 64       # small for demo (paper uses 512)
num_heads = 8
d_ff = 256         # paper uses 2048
num_layers = 6

model = Transformer(
    src_vocab_size=src_vocab,
    tgt_vocab_size=tgt_vocab,
    d_model=d_model,
    num_heads=num_heads,
    d_ff=d_ff,
    num_layers=num_layers
)

# Dummy data: batch of 2 sentences
src = torch.randint(0, src_vocab, (2, 12))   # source: batch=2, seq_len=12
tgt = torch.randint(0, tgt_vocab, (2, 10))   # target: batch=2, seq_len=10

# Causal mask for target
tgt_len = tgt.shape[1]
tgt_mask = torch.tril(torch.ones(tgt_len, tgt_len)).unsqueeze(0).unsqueeze(0)

# Forward pass
logits = model(src, tgt, tgt_mask=tgt_mask)

print(f"Source tokens:  {src.shape}  (batch=2, src_len=12)")
print(f"Target tokens:  {tgt.shape}  (batch=2, tgt_len=10)")
print(f"Output logits:  {logits.shape}  (batch=2, tgt_len=10, tgt_vocab={tgt_vocab})")
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"\nPaper's base model had ~65M parameters (d_model=512, d_ff=2048)")

In [ ]:
# Trace the data flow through each component
print("=== Data Flow Through the Transformer ===")
print()

# Encoder side
src_emb = model.src_embedding(src) * math.sqrt(d_model)
print(f"1. Source embedding:     {src.shape} → {src_emb.shape}")

src_pe = model.positional_encoding(src_emb)
print(f"2. + Positional encoding: {src_pe.shape}")

enc_out = model.encoder(src_pe)
print(f"3. Encoder output:       {enc_out.shape}")

# Decoder side
tgt_emb = model.tgt_embedding(tgt) * math.sqrt(d_model)
print(f"4. Target embedding:     {tgt.shape} → {tgt_emb.shape}")

tgt_pe = model.positional_encoding(tgt_emb)
print(f"5. + Positional encoding: {tgt_pe.shape}")

dec_out = model.decoder(tgt_pe, enc_out, tgt_mask=tgt_mask)
print(f"6. Decoder output:       {dec_out.shape}")

logits = model.fc_out(dec_out)
print(f"7. Final logits:         {logits.shape}")

probs = F.softmax(logits, dim=-1)
print(f"8. Probabilities:        {probs.shape}")
print(f"   (sum per position = {probs[0, 0].sum().item():.4f})")

## 8. Key Insights from the Paper

### Why Self-Attention Beats RNNs (Paper Table 1)

| Property | Self-Attention | RNN | CNN |
|----------|---------------|-----|-----|
| Complexity per layer | $O(n^2 \cdot d)$ | $O(n \cdot d^2)$ | $O(k \cdot n \cdot d^2)$ |
| Sequential ops | $O(1)$ | $O(n)$ | $O(1)$ |
| Max path length | $O(1)$ | $O(n)$ | $O(\log_k n)$ |

Key advantages:
- **Parallelizable:** All positions computed simultaneously ($O(1)$ sequential ops vs $O(n)$ for RNN)
- **Short paths:** Any two positions are directly connected (max path length $O(1)$)
- For typical NLP: $n < d$, so $O(n^2 d) < O(n d^2)$

### Training Details
- **Optimizer:** Adam with $\beta_1=0.9, \beta_2=0.98$
- **Learning rate warmup:** Linearly increase for 4000 steps, then decay
- **Label smoothing:** $\epsilon_{ls} = 0.1$ (hurts perplexity but improves BLEU)
- **Dropout:** 0.1 on residual connections and attention weights

### Results
- English→German: **28.4 BLEU** (new SOTA, +2.0 over previous best)
- English→French: **41.0 BLEU** (new SOTA)
- Training cost: 3.5 days on 8 P100 GPUs (fraction of competing models)

### Exercise 1: Change d_model

Create Transformers with d_model = 32, 64, 128 and compare parameter counts.

In [ ]:
# Exercise 1
for d in [32, 64, 128, 256, 512]:
    m = Transformer(src_vocab_size=1000, tgt_vocab_size=1000,
                    d_model=d, num_heads=8, d_ff=d*4, num_layers=6)
    params = sum(p.numel() for p in m.parameters())
    print(f"d_model={d:3d}, d_ff={d*4:4d} → {params:>10,} parameters")

print("\nParameters grow roughly quadratically with d_model!")

### Exercise 2: Visualize Positional Encodings at Different Dimensions

In [ ]:
# Exercise 2: Plot individual PE dimensions
pe_plot = PositionalEncoding(d_model=64, max_len=200)
pe_vals = pe_plot.pe[0].numpy()

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
dims_to_plot = [0, 1, 10, 11]  # sin/cos pairs

for ax, dim in zip(axes.flat, dims_to_plot):
    ax.plot(pe_vals[:100, dim])
    kind = 'sin' if dim % 2 == 0 else 'cos'
    freq_idx = dim // 2
    ax.set_title(f'Dimension {dim} ({kind}, i={freq_idx})')
    ax.set_xlabel('Position')
    ax.set_ylabel('Value')
    ax.grid(True, alpha=0.3)

plt.suptitle('Positional Encoding: Low dimensions = high frequency, High dimensions = low frequency', y=1.02)
plt.tight_layout()
plt.show()
print("Lower dimensions oscillate quickly (capture fine position differences).")
print("Higher dimensions oscillate slowly (capture coarse position information).")

## Summary: What Made This Paper Revolutionary

1. **Eliminated recurrence entirely** — no more sequential bottleneck. Training is massively parallelizable.

2. **Self-attention connects everything** — any token can directly attend to any other token, regardless of distance. No more vanishing gradients across long sequences.

3. **Multi-head attention** — the model can simultaneously attend to information from different representation subspaces at different positions.

4. **Simplicity and generality** — the same architecture works for any sequence-to-sequence task, and has since been adapted for language modeling (GPT), understanding (BERT), vision (ViT), and more.

---

## Congratulations!

You've built a complete Transformer from scratch! Here's what to explore next:

- **BERT** (Devlin et al., 2018) — encoder-only Transformer for understanding
- **GPT** (Radford et al., 2018) — decoder-only Transformer for generation
- **Vision Transformer (ViT)** — applying Transformers to images
- **Try training** your Transformer on a small translation dataset!
- Read the original paper: [arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)